PROMPT MAESTRO

In [ ]:
import os
import time
import shutil
from datetime import datetime
from google.colab import drive

# 1. Montar Google Drive en el entorno de Colab
drive.mount('/content/drive')

# Rutas de trabajo según la estructura del taller
BASE_DIR = '/content/drive/MyDrive/TechDistribuidora'
INPUT_DIR = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
PROCESADOS_DIR = os.path.join(BASE_DIR, 'procesados')
LOG_FILE = os.path.join(BASE_DIR, 'proceso.log')

# Asegurar existencia de las carpetas requeridas
for folder in [INPUT_DIR, OUTPUT_DIR, PROCESADOS_DIR]:
    os.makedirs(folder, exist_ok=True)


def registrar_log(mensaje):
    """Escribe en consola y añade la línea en proceso.log con marca de tiempo persistente."""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    registro = f"{timestamp} - {mensaje}\n"
    print(registro.strip())

    with open(LOG_FILE, 'a', encoding='utf-8') as log:
        log.write(registro)
        log.flush()
        os.fsync(log.fileno())


def procesar_archivos_base():
    """Escanea input/, filtra .csv, copia a output/ y mueve el original a procesados/ para evitar reprocesamiento."""
    # 3. Procesar solo archivos .csv
    archivos = [
        f for f in os.listdir(INPUT_DIR)
        if f.lower().endswith('.csv') and os.path.isfile(os.path.join(INPUT_DIR, f))
    ]

    for archivo in archivos:
        ruta_origen = os.path.join(INPUT_DIR, archivo)
        ruta_salida = os.path.join(OUTPUT_DIR, archivo)
        ruta_historial = os.path.join(PROCESADOS_DIR, archivo)

        # 6. No procesar dos veces el mismo archivo
        if os.path.exists(ruta_historial):
            registrar_log(f"ADVERTENCIA: El archivo '{archivo}' ya fue atendido previamente en 'procesados/'. Omitiendo.")
            # Se retira de input para no trabar el ciclo de sondeo
            os.remove(ruta_origen)
            continue

        registrar_log(f"Iniciando procesamiento de archivo: '{archivo}'")

        # 4. Copiar a la subcarpeta output/
        shutil.copy2(ruta_origen, ruta_salida)
        registrar_log(f"Copia exitosa de '{archivo}' hacia '{OUTPUT_DIR}'")

        # Mover original a procesados/ para garantizar que no se procese de nuevo
        shutil.move(ruta_origen, ruta_historial)
        registrar_log(f"Archivo original '{archivo}' archivado en '{PROCESADOS_DIR}'")


# 2. Monitoreo continuo cada 10 segundos
INTERVALO = 10
registrar_log("=== SERVICIO FILE TRANSFER (VERSIÓN BASE) INICIADO ===")
print(f"Monitoreando '{INPUT_DIR}' cada {INTERVALO} segundos. Detén la celda para parar.\n")

try:
    while True:
        procesar_archivos_base()
        time.sleep(INTERVALO)
except KeyboardInterrupt:
    registrar_log("=== SERVICIO DETENIDO MANUALMENTE ===")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-10-18 21:18:03 - === SERVICIO FILE TRANSFER (VERSIÓN BASE) INICIADO ===
Monitoreando '/content/drive/MyDrive/TechDistribuidora/input' cada 10 segundos. Detén la celda para parar.

2026-10-18 21:18:03 - ADVERTENCIA: El archivo 'pedidos_validos.csv' ya fue atendido previamente en 'procesados/'. Omitiendo.
2026-10-18 21:19:04 - === SERVICIO DETENIDO MANUALMENTE ===


PROMPT CON ITERACION

In [ ]:
import os
import time
import shutil
from datetime import datetime
import pandas as pd
from google.colab import drive

# 1. Montaje de Google Drive
drive.mount('/content/drive')

# Definición de rutas base y directorios del patrón File Transfer[span_0](start_span)[span_0](end_span)
BASE_DIR = '/content/drive/MyDrive/TechDistribuidora'
INPUT_DIR = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
ERROR_DIR = os.path.join(BASE_DIR, 'error')
PROCESADOS_DIR = os.path.join(BASE_DIR, 'procesados')
LOG_FILE = os.path.join(BASE_DIR, 'proceso.log')

# Crear la estructura de carpetas si no existen[span_1](start_span)[span_1](end_span)
for path in [INPUT_DIR, OUTPUT_DIR, ERROR_DIR, PROCESADOS_DIR]:
    os.makedirs(path, exist_ok=True)


def escribir_log(mensaje, nivel="INFO"):
    """
    Escribe en consola y en proceso.log forzando la sincronización física
    inmediata con Google Drive (evita retención en el búfer de Colab)[span_2](start_span)[span_2](end_span).
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    linea = f"{timestamp} [{nivel}] {mensaje}\n"

    print(linea.strip())

    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(linea)
        f.flush()
        os.fsync(f.fileno())


COLUMNAS_ESPERADAS = ['id', 'cliente', 'producto', 'cantidad', 'total']


def validar_csv(ruta):
    """
    Valida la existencia de columnas y los tipos de datos por fila:
    - cantidad > 0
    - total numérico y no nulo[span_3](start_span)[span_3](end_span)
    """
    fallos = []
    try:
        df = pd.read_csv(ruta)
    except Exception as e:
        return False, None, [f"Error sintaxis CSV: {str(e)}"]

    faltantes = [c for c in COLUMNAS_ESPERADAS if c not in df.columns]
    if faltantes:
        return False, None, [f"Columnas ausentes en cabecera: {faltantes}"]

    for idx, fila in df.iterrows():
        linea = idx + 2

        # Validar cantidad
        try:
            cant = float(fila['cantidad'])
            if cant <= 0:
                fallos.append(f"Línea {linea}: 'cantidad' debe ser mayor a 0 (recibido: {fila['cantidad']})")
        except (ValueError, TypeError):
            fallos.append(f"Línea {linea}: 'cantidad' no numérica (recibido: {fila['cantidad']})")

        # Validar total
        try:
            tot = float(fila['total'])
            if pd.isna(tot):
                fallos.append(f"Línea {linea}: 'total' es nulo o vacío")
        except (ValueError, TypeError):
            fallos.append(f"Línea {linea}: 'total' no numérico (recibido: {fila['total']})")

    return len(fallos) == 0, df, fallos


def transformar(df):
    """
    Transformación de negocio:
    - Nombre del cliente en MAYÚSCULAS[span_4](start_span)[span_4](end_span)
    - Firma técnica: cálculo de base imponible y marca temporal de auditoría[span_5](start_span)[span_5](end_span)
    """
    df_out = df.copy()
    df_out['cliente'] = df_out['cliente'].astype(str).str.strip().str.upper()
    df_out['producto'] = df_out['producto'].astype(str).str.strip()
    df_out['cantidad'] = df_out['cantidad'].astype(int)
    df_out['total'] = df_out['total'].astype(float).round(2)
    df_out['subtotal_neto'] = (df_out['total'] / 1.15).round(2)
    df_out['fecha_procesamiento'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return df_out


def procesar_archivo(archivo):
    ruta_in = os.path.join(INPUT_DIR, archivo)
    ruta_historico = os.path.join(PROCESADOS_DIR, archivo)

    # --------------------------------------------------------------------------
    # CONTROL DE DUPLICADOS: No procesar dos veces el mismo archivo[span_6](start_span)[span_6](end_span)
    # Verifica si el archivo ya fue atendido con éxito en una ejecución previa
    # --------------------------------------------------------------------------
    if os.path.exists(ruta_historico):
        escribir_log(
            f"RECHAZO POR DUPLICIDAD: El archivo '{archivo}' ya fue procesado anteriormente y existe en 'procesados/'.",
            "WARNING"
        )
        # Se mueve a error con prefijo para no sobrescribir el histórico ni bloquear la cola input/
        timestamp_marca = datetime.now().strftime('%Y%m%d_%H%M%S')
        destino_descarte = os.path.join(ERROR_DIR, f"duplicado_{timestamp_marca}_{archivo}")
        shutil.move(ruta_in, destino_descarte)
        escribir_log(f"Archivo duplicado desviado a: {destino_descarte}", "INFO")
        return

    escribir_log(f"Procesando archivo entrante: '{archivo}'", "INFO")

    es_valido, df, fallos = validar_csv(ruta_in)

    if es_valido:
        escribir_log(f"Validación exitosa de '{archivo}'. Generando transformación...", "INFO")
        df_t = transformar(df)

        # Guardar en output/[span_7](start_span)[span_7](end_span)
        ruta_out = os.path.join(OUTPUT_DIR, f"procesado_{archivo}")
        df_t.to_csv(ruta_out, index=False, encoding='utf-8')

        # Mover original a procesados/ para consumo idempotente[span_8](start_span)[span_8](end_span)
        shutil.move(ruta_in, ruta_historico)
        escribir_log(f"Completado: guardado en output/procesado_{archivo} y movido original a procesados/{archivo}", "INFO")
    else:
        escribir_log(f"Validación fallida para '{archivo}'. Fallos detectados: {len(fallos)}", "WARNING")
        for f in fallos:
            escribir_log(f"  -> Detalle: {f}", "WARNING")

        # Mover archivo con inconsistencias a error/[span_9](start_span)[span_9](end_span)[span_10](start_span)[span_10](end_span)
        shutil.move(ruta_in, os.path.join(ERROR_DIR, archivo))
        escribir_log(f"Archivo trasladado a: error/{archivo}", "INFO")


# Bucle principal de sondeo (Polling continuo cada 10 segundos)[span_11](start_span)[span_11](end_span)[span_12](start_span)[span_12](end_span)
POLL_INTERVAL = 10
escribir_log("=== SERVICIO FILE TRANSFER INICIADO ===", "INFO")
print(f"Monitoreando la carpeta '{INPUT_DIR}' cada {POLL_INTERVAL} segundos. Detén la celda para finalizar.\n")

try:
    while True:
        # Filtrar exclusivamente archivos con extensión .csv[span_13](start_span)[span_13](end_span)
        csvs = [
            f for f in os.listdir(INPUT_DIR)
            if f.lower().endswith('.csv') and os.path.isfile(os.path.join(INPUT_DIR, f))
        ]

        for csv_file in csvs:
            procesar_archivo(csv_file)

        time.sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    escribir_log("=== SERVICIO DETENIDO POR EL USUARIO ===", "INFO")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-10-18 21:24:16 [INFO] === SERVICIO FILE TRANSFER INICIADO ===
Monitoreando la carpeta '/content/drive/MyDrive/TechDistribuidora/input' cada 10 segundos. Detén la celda para finalizar.

2026-10-18 21:24:46 [INFO] Procesando archivo entrante: 'pedidos_validos.csv'
2026-10-18 21:24:46 [INFO] Validación exitosa de 'pedidos_validos.csv'. Generando transformación...
2026-10-18 21:24:46 [INFO] Completado: guardado en output/procesado_pedidos_validos.csv y movido original a procesados/pedidos_validos.csv
2026-10-18 21:29:36 [INFO] Procesando archivo entrante: 'pedidos_invalidos.csv'
2026-10-18 21:29:37 [WARNING] Validación fallida para 'pedidos_invalidos.csv'. Fallos detectados: 6
2026-10-18 21:29:37 [WARNING]   -> Detalle: Línea 2: 'cantidad' debe ser mayor a 0 (recibido: -3)
2026-10-18 21:29:37 [WARNING]   -> Detalle: Línea 3: 'total' no numérico (recibido: ER

GENERACION DE CVS

In [ ]:
# Script para crear los archivos CSV de prueba localmente en el entorno
csv_valido = """id,cliente,producto,cantidad,total
101,Juan Perez,Laptop Dell Inspiron,2,1850.50
102,mariana lopez,Monitor LG 24 Pulgadas,4,620.00
103,Carlos Davila,Teclado Mecanico RGB,5,375.25
104,ana torres,Mouse Inalambrico Logitech,10,250.00
105,DISTRIBUIDORA NORTE,Impresora Multifuncion Epson,3,810.90
106,Sofia Mendoza,Disco Solido Kingston 1TB,6,450.00"""

csv_invalido = """id,cliente,producto,cantidad,total
201,Empresa Alfa,Impresora Laser HP,-3,450.00
202,Comercial Beta,Memoria RAM 16GB Kingston,2,ERROR_PRECIO
203,Pedro Gomez,Disco SSD Crucial 1TB,0,120.00
204,Laura Silva,Router Wi-Fi 6 TP-Link,1,NULL
205,Andres Castro,Switch Gigabit 8 Puertos,Cinco,150.00
206,Maria Belen,Monitor Curvo Samsung,2,"""

# Guardar archivos
with open('pedidos_validos.csv', 'w', encoding='utf-8') as f:
    f.write(csv_valido)

with open('pedidos_invalidos.csv', 'w', encoding='utf-8') as f:
    f.write(csv_invalido)

print("Archivos 'pedidos_validos.csv' y 'pedidos_invalidos.csv' generados con éxito.")
